# Proyecto AEMET - Resumen de Extracción, Transformación y Carga (ETL) de Datos Climatológicos
Este notebook consolida y explica paso a paso todo el flujo de trabajo desarrollado para interactuar con la API de la AEMET (Agencia Estatal de Meteorología), procesar y limpiar la información, y exportar un dataset histórico listo para análisis o entrenamiento de modelos.

---

## 📋 Objetivos de este Notebook
1. **Autenticación e Integración:** Conectar con la API de AEMET OpenData utilizando una API Key almacenada de forma segura en un archivo `.env` local.
2. **Descarga del Catálogo de Estaciones:** Obtener y mostrar el maestro de estaciones meteorológicas operativas en España.
3. **Consulta de Datos de Prueba:** Descargar un subconjunto de datos diarios para una estación específica en un rango de fechas reducido para validar la conexión.
4. **Limpieza y Modelado de Datos (ETL):** Procesar la respuesta JSON de la API, reemplazando caracteres incorrectos, convirtiendo tipos de datos (strings a floats con coma, datetime) y manejando nulos de forma robusta.
5. **Descarga Masiva e Histórica:** Implementar un bucle para extraer datos históricos de los últimos años, gestionando de forma proactiva los límites de velocidad (rate limiting) de la API mediante retardos estratégicos.
6. **Exportación de Datos:** Consolidar el dataset completo en archivos Excel y CSV en el espacio local del usuario.
7. **Visualización Resumen:** Crear un gráfico de control que muestre la evolución del clima a lo largo del tiempo para validar los datos visualmente.

In [ ]:
# Importamos las librerías necesarias
import os
import zipfile
import sys
import time
import json
import calendar
import subprocess
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv

# Tuve que meter este parche raro de zipfile porque me daba error al cerrar archivos excel en python 3.14+
# 'ValueError: seek of closed file'. Lo saqué de StackOverflow
try:
    _orig_zipfile_del = zipfile.ZipFile.__del__
    def _safe_zipfile_del(self):
        try:
            _orig_zipfile_del(self)
        except (ValueError, AttributeError):
            pass
    zipfile.ZipFile.__del__ = _safe_zipfile_del
except AttributeError:
    pass


# Función para comprobar e instalar automáticamente las librerías si no están instaladas
def asegurar_libreria(nombre_paquete, nombre_import=None):
    if nombre_import is None:
        nombre_import = nombre_paquete
    try:
        __import__(nombre_import)
    except ImportError:
        print(f"No tienes instalado '{nombre_paquete}'. Instalando con pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", nombre_paquete])
        print(f"'{nombre_paquete}' instalada correctamente.")

asegurar_libreria("openpyxl")
asegurar_libreria("matplotlib", "matplotlib.pyplot")
asegurar_libreria("seaborn")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("Librerías listas!")

## Paso 1: Configuración del Entorno y Carga de Variables Seguras

Para evitar exponer credenciales en el código fuente, la API Key de AEMET se almacena en el archivo `.env` de esta carpeta. Usamos la biblioteca `python-dotenv` para cargar el entorno de forma segura, apuntando específicamente al directorio de este notebook.

In [ ]:
# Buscamos el archivo .env para la API KEY de AEMET
BASE_DIR = Path().resolve()
env_path = BASE_DIR / '.env'

print("Buscando el archivo .env en:", env_path)

if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print("El archivo .env se ha cargado.")
else:
    print("Ojo: No se encontró el .env en esta carpeta. Probamos con el entorno general.")
    load_dotenv()

# Sacamos la API Key
API_KEY = os.getenv('AEMET_API_KEY')
if not API_KEY:
    raise RuntimeError('No se encontró AEMET_API_KEY en las variables de entorno.')
else:
    # Mostramos un trozo por seguridad
    print(f"API Key encontrada: {API_KEY[:6]}...{API_KEY[-6:]}")

## Paso 2: El Protocolo de Conexión de AEMET OpenData

La API de AEMET funciona bajo un modelo de **"doble llamada"**:
1. Enviamos nuestra consulta con la clave `api_key` en las cabeceras (headers) de la solicitud HTTP GET.
2. AEMET valida la solicitud y nos responde con un JSON de metadatos que contiene una URL temporal en el campo `datos`.
3. Hacemos una segunda solicitud `GET` a esa URL temporal (esta vez sin cabeceras de autorización) para descargar los datos reales del sensor en formato JSON.

Definimos el diccionario de cabeceras que utilizaremos en la primera llamada de autenticación.

In [ ]:
# Preparamos las cabeceras para la petición
headers = {
    'cache-control': "no-cache",
    'api_key': API_KEY,
    'accept': "application/json"
}

print("Cabeceras listas para usar.")

## Paso 3: Obtener el Inventario Completo de Estaciones Climatológicas

Para realizar cualquier consulta histórica, necesitamos conocer el identificador único (`indicativo` o `idema`) de la estación meteorológica. El endpoint `/api/valores/climatologicos/inventarioestaciones/todasestaciones` nos devuelve el catálogo completo de las estaciones activas en España.

In [ ]:
# Función para traer el listado de todas las estaciones
def obtener_inventario_estaciones():
    url_base = "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones"
    
    print("Pidiendo el inventario a la API:", url_base)
    try:
        response = requests.get(url_base, headers=headers)
        if response.status_code == 200:
            meta_datos = response.json()
            print(f"Respuesta API: {meta_datos.get('descripcion')} (Estado: {meta_datos.get('estado')})")
            
            if meta_datos.get('estado') == 200:
                url_datos = meta_datos.get('datos')
                print("Tenemos enlace temporal, descargando los datos reales...")
                
                # Segunda llamada
                datos_response = requests.get(url_datos)
                if datos_response.status_code == 200:
                    return datos_response.json()
                else:
                    print("Error al descargar datos finales, código:", datos_response.status_code)
            else:
                print("Error en AEMET:", meta_datos.get('descripcion'))
        else:
            print("Error de red en la primera llamada:", response.status_code)
    except Exception as e:
        print("Ha ocurrido una excepción:", e)
    return None

# Probamos a traer las estaciones
estaciones = obtener_inventario_estaciones()

if estaciones:
    print(f"Total estaciones cargadas: {len(estaciones)}")
    df_estaciones = pd.DataFrame(estaciones)
    print("Primeras 5 estaciones:")
    display(df_estaciones.head(5))
else:
    print("No se pudo obtener el inventario. Revisa la clave API.")

## Paso 4: Consulta de Prueba de Valores Climatológicos Diarios

Para verificar el flujo de descarga de datos de sensores diarios, consultaremos los datos del 1 al 15 de mayo de 2026 para la estación de **Palma de Mallorca (Puerto)** (indicativo o idema: `"B228"`).

El formato del endpoint requerido es:
`/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/estacion/{idema}`

In [ ]:
# Petición para una sola estación (para pruebas)
def obtener_valores_climaticos(fecha_ini_utc, fecha_fin_utc, idema):
    url_base = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_utc}/fechafin/{fecha_fin_utc}/estacion/{idema}"
    
    try:
        response = requests.get(url_base, headers=headers)
        if response.status_code == 200:
            meta_datos = response.json()
            if meta_datos.get('estado') == 200:
                url_final = meta_datos.get('datos')
                datos_response = requests.get(url_final)
                if datos_response.status_code == 200:
                    return datos_response.json()
                else:
                    print("Error descargando los datos climatológicos:", datos_response.status_code)
            else:
                return None
        else:
            print("Error de red:", response.status_code)
    except Exception as e:
        print("Excepción al consultar clima:", e)
    return None


# Petición para todas las estaciones con reintentos y esperas (backoff) si falla la API
def obtener_valores_climaticos_todas(fecha_ini_utc, fecha_fin_utc, max_retries=15, base_delay=5.0):
    url_base = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_utc}/fechafin/{fecha_fin_utc}/todasestaciones"
    delay = base_delay
    for intento in range(max_retries):
        try:
            response = requests.get(url_base, headers=headers)
            
            # Si no tenemos permiso o no existe el endpoint, paramos
            if response.status_code in [401, 403, 404]:
                print(f"Error grave {response.status_code}, no se puede reintentar.")
                return None
                
            if response.status_code == 200:
                meta_datos = response.json()
                estado = meta_datos.get('estado')
                
                if estado == 200:
                    url_final = meta_datos.get('datos')
                    datos_response = requests.get(url_final)
                    if datos_response.status_code == 200:
                        return datos_response.json()
                    elif datos_response.status_code in [401, 403, 404]:
                        print(f"Error grave en datos {datos_response.status_code}, no reintentamos.")
                        return None
                    else:
                        print(f"Intento {intento+1} fallido al descargar datos: {datos_response.status_code}")
                elif estado in [401, 403, 404]:
                    print(f"Error API: Estado {estado}: {meta_datos.get('descripcion')}. No se reintentará.")
                    return None
                elif estado == 429:
                    print(f"Intento {intento+1}: La API dice Estado 429 (Límite superado).")
                else:
                    print(f"Intento {intento+1}: API devolvió Estado {estado}: {meta_datos.get('descripcion')}")
            elif response.status_code == 429:
                print(f"Intento {intento+1}: HTTP 429 (Muchas peticiones).")
            else:
                print(f"Intento {intento+1}: Código de red {response.status_code}")
                
        except Exception as e:
            print(f"Intento {intento+1}: Error de red/conexión: {e}")
            
        if intento < max_retries - 1:
            print(f"Esperando {delay} segundos antes de reintentar...")
            time.sleep(delay)
            delay *= 2.0
            
    print(f"No se pudo descargar el bloque tras {max_retries} intentos.")
    return None

# Hacemos una prueba con Palma de Mallorca (B228)
idema_prueba = "B228"
ini_prueba = "2026-05-01T00:00:00UTC"
fin_prueba = "2026-05-15T00:00:00UTC"

print("Probando descarga para Palma de Mallorca...")
climaticos_prueba = obtener_valores_climaticos(ini_prueba, fin_prueba, idema_prueba)

if climaticos_prueba:
    print(f"¡Funciona! Hemos descargado {len(climaticos_prueba)} filas diarias.")
    print("Ejemplo del primer registro en crudo:")
    print(json.dumps(climaticos_prueba[0], indent=2, ensure_ascii=False))
else:
    print("No se han recibido datos. Revisa fechas y tu API Key.")

## Paso 5: Procesamiento y Limpieza de Datos (Pipeline ETL)

Los datos crudos de AEMET vienen completamente en formato de texto (strings). Además, la representación decimal utiliza la coma española (`,`) en lugar del punto (`.`), lo cual impide operaciones aritméticas en Pandas.

Implementamos la función `limpiar_y_cargar_datos` que realiza el siguiente flujo de transformación:
1. Extrae los campos de interés de cada registro.
2. Convierte el campo `fecha` a tipo `datetime` de Pandas.
3. Normaliza las representaciones numéricas: reemplaza las comas por puntos en variables continuas y las convierte a tipo `float32` o `int` de forma segura.
4. Convierte las horas de temperaturas/presiones máximas y mínimas a objetos `datetime.time`, gestionando de forma robusta valores de texto anómalos como 'Varias' o nulos.
5. Limpia el campo de precipitación (`prec`), convirtiendo el valor especial "Ip" (inapreciable) a `0.0` para poder realizar cálculos y sumas acumuladas consistentes.

In [ ]:
# Limpieza y preparación de los datos obtenidos
def limpiar_y_cargar_datos(datos_json):
    if not datos_json:
        return pd.DataFrame()
        
    datos_lista = []
    for dia in datos_json:
        datos_lista.append({
            "fecha": dia.get("fecha"),
            "indicativo": dia.get("indicativo"),
            "nombre": dia.get("nombre"),
            "provincia": dia.get("provincia"),
            "altitud": dia.get("altitud"),
            "tmed": dia.get("tmed"),
            "prec": dia.get("prec", "0,00"),
            "tmin": dia.get("tmin"),
            "horatmin": dia.get("horatmin"),
            "tmax": dia.get("tmax"),
            "horatmax": dia.get("horatmax"),
            "dir": dia.get("dir"),
            "velmedia": dia.get("velmedia"),
            "racha": dia.get("racha"),
            "horaracha": dia.get("horaracha"),
            "sol": dia.get("sol"),
            "presMax": dia.get("presMax"),
            "horaPresMax": dia.get("horaPresMax"),
            "presMin": dia.get("presMin"),
            "horaPresMin": dia.get("horaPresMin"),
            "hrMedia": dia.get("hrMedia"),
            "hrMax": dia.get("hrMax"),
            "horaHrMax": dia.get("horaHrMax"),
            "hrMin": dia.get("hrMin"),
            "horaHrMin": dia.get("horaHrMin")
        })

    df = pd.DataFrame(datos_lista)

    # 1. Ajustes básicos de tipos de datos
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["indicativo"] = df["indicativo"].astype(str)
    df["nombre"] = df["nombre"].astype(str)
    df["provincia"] = df["provincia"].astype(str)
    df["altitud"] = pd.to_numeric(df["altitud"], errors="coerce")

    # Función para pasar números con coma (español) a float con punto
    def a_float(columna):
        if columna is None:
            return np.nan
        return columna.astype(str).str.replace(",", ".").replace("", "nan").astype(np.float32)

    # 2. Convertimos las columnas numéricas
    columnas_numericas = ["tmed", "tmin", "tmax", "velmedia", "racha", "sol", "presMax", "presMin", "hrMedia", "hrMax", "hrMin"]
    for col in columnas_numericas:
        if col in df.columns:
            df[col] = a_float(df[col])

    # 3. Arreglamos la precipitación (la API a veces devuelve "Ip" que significa inapreciable, la ponemos en 0.0)
    if "prec" in df.columns:
        df["prec"] = df["prec"].astype(str).str.replace("Ip", "0.0").str.replace(",", ".").replace("", "0.0")
        df["prec"] = pd.to_numeric(df["prec"], errors="coerce").astype(np.float32)

    # 4. Limpiamos horas y las pasamos a tipo time de python
    columnas_tiempo = ["horatmin", "horatmax", "horaHrMax", "horaHrMin"]
    for col in columnas_tiempo:
        if col in df.columns:
            df[col] = df[col].astype(str).replace(["Varias", "nan", "None", ""], np.nan)
            df[col] = pd.to_datetime(df[col], format="%H:%M", errors="coerce").dt.time

    # 5. Otras horas que vienen solo con la hora
    columnas_hora_sola = ["horaPresMax", "horaPresMin"]
    for col in columnas_hora_sola:
        if col in df.columns:
            df[col] = df[col].astype(str).replace(["Varias", "nan", "None", ""], np.nan)
            df[col] = pd.to_datetime(df[col], format="%H", errors="coerce").dt.time

    if "dir" in df.columns:
        df["dir"] = df["dir"].astype(str)
    if "horaracha" in df.columns:
        df["horaracha"] = df["horaracha"].astype(str)

    return df

# Probamos a limpiar los datos de prueba
df_prueba = limpiar_y_cargar_datos(climaticos_prueba)
print("Limpieza de prueba terminada!")
print("Tipos del DataFrame:")
df_prueba.info()
print("Muestra:")
display(df_prueba.head(5))

## Paso 6: Extracción Histórica Masiva con Control de Rate Limiting

Para obtener un dataset consistente e histórico, se implementa un bucle recursivo que descarga datos año por año y mes por mes.

**⚠️ IMPORTANTE - Límites de la API de AEMET:**
AEMET limita estrictamente la cantidad de llamadas simultáneas. Si no se manejan pausas, la API responderá con errores `HTTP 429` (Demasiadas peticiones) o bloqueará el token de usuario.
Para evitar esto, implementamos:
1. **Pausa de 60 segundos** entre cada año consultado.
2. **Pausa de 1.5 segundos** entre cada mes consultado.
3. **Pausa de 40 minutos** de seguridad si el tiempo total de ejecución supera los 5 minutos de forma continua.
4. **Saltado dinámico de meses futuros** para evitar llamadas inútiles sobre periodos que aún no han ocurrido.

*Nota de control:* Por defecto, esta descarga está pre-configurada para extraer los **últimos 3 años** (2024-2026) para optimizar el tiempo de prueba de este notebook. Si deseas obtener la serie histórica completa de los últimos 10 años, simplemente modifica la variable `AÑOS_A_CONSULTAR = 10`.

In [ ]:
from datetime import datetime, timedelta
import gc

# Ajustamos las fechas para la descarga histórica (10 años en bloques)
fecha_fin_global = datetime(2026, 5, 30)
DIAS_A_CONSULTAR = 3650  # 10 años
fecha_inicio_global = fecha_fin_global - timedelta(days=DIAS_A_CONSULTAR - 1)

# Carpetas de salida
OUTPUT_CSV_DIR = BASE_DIR / 'sheets' / 'csv'
OUTPUT_XLSX_DIR = BASE_DIR / 'sheets' / 'xlsx'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_XLSX_DIR.mkdir(parents=True, exist_ok=True)

año_inicio = fecha_inicio_global.year
año_fin = fecha_fin_global.year

print(f"Empezando la descarga masiva por años desde {año_inicio} hasta {año_fin}...")

for anio in range(año_fin, año_inicio - 1, -1):
    print("\n-----------------------------------------")
    print(f"Procesando el año: {anio}")
    print("-----------------------------------------")
    
    ini_anio = max(datetime(anio, 1, 1), fecha_inicio_global)
    fin_anio = min(datetime(anio, 12, 31), fecha_fin_global)
    
    df_anio = pd.DataFrame()
    fecha_actual = fin_anio
    
    lote_size = 15  # Descargamos en bloques de 15 días
    dias_anio_procesados = 0
    total_dias_anio = (fin_anio - ini_anio).days + 1
    
    print(f"Fechas para {anio}: de {ini_anio.strftime('%Y-%m-%d')} a {fin_anio.strftime('%Y-%m-%d')} ({total_dias_anio} días)")
    
    while dias_anio_procesados < total_dias_anio:
        dias_restantes = total_dias_anio - dias_anio_procesados
        dias_lote = min(lote_size, dias_restantes)
        
        fecha_ini_lote = fecha_actual - timedelta(days=dias_lote - 1)
        
        fecha_ini_str = fecha_ini_lote.strftime("%Y-%m-%dT00:00:00UTC")
        fecha_fin_str = fecha_actual.strftime("%Y-%m-%dT23:59:59UTC")
        
        print(f"Descargando bloque: {fecha_ini_str} a {fecha_fin_str}...")
        
        # Esperamos 2 segundos entre descargas para no pasarnos del límite de la API
        print("Pausa de 2 segundos...")
        time.sleep(2.0)
        
        datos_json = obtener_valores_climaticos_todas(fecha_ini_str, fecha_fin_str)
        
        if datos_json:
            df_lote = limpiar_y_cargar_datos(datos_json)
            df_anio = pd.concat([df_anio, df_lote], ignore_index=True)
            print(f"Ok, bloque descargado. Registros nuevos: {len(df_lote)}")
        else:
            print(f"Error en bloque de {fecha_ini_str} a {fecha_fin_str}")
            
        dias_anio_procesados += dias_lote
        fecha_actual = fecha_ini_lote - timedelta(days=1)
        
    # Guardamos el año completo y limpiamos memoria RAM
    if not df_anio.empty:
        # Ordenar por fecha cronológica más antigua a más reciente
        if 'fecha' in df_anio.columns:
            df_anio = df_anio.sort_values(by='fecha', ascending=True)
            
        csv_path = OUTPUT_CSV_DIR / f'climaticos_{anio}.csv'
        excel_path = OUTPUT_XLSX_DIR / f'climaticos_{anio}.xlsx'
        
        print(f"Guardando ficheros para el año {anio}...")
        try:
            # Guardamos en CSV
            df_anio.to_csv(csv_path, index=False)
            print(f"Guardado CSV en: {csv_path}")
            
            # Guardamos en Excel con un ExcelWriter
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                df_anio.to_excel(writer, index=False)
            print(f"Guardado Excel en: {excel_path}")
        except Exception as e:
            print(f"Error al guardar los archivos de {anio}: {e}")
            
        # Forzamos la limpieza de RAM
        del df_anio
        gc.collect()
    else:
        print(f"No hay datos para el año {anio}, no guardamos nada.")

print("\n¡Descarga y exportación completadas!")

## Paso 7: Verificación de los Archivos Físicos Generados

Para validar que la exportación anual y segmentada se ha completado correctamente en disco, en esta sección listamos el contenido de los directorios de salida  y , mostrando el tamaño ocupado por cada archivo generado.

In [ ]:
# Comprobamos los archivos creados en el disco y su tamaño
OUTPUT_CSV_DIR = BASE_DIR / 'sheets' / 'csv'
OUTPUT_XLSX_DIR = BASE_DIR / 'sheets' / 'xlsx'

print("Archivos CSV creados:")
if OUTPUT_CSV_DIR.exists():
    for f in sorted(OUTPUT_CSV_DIR.glob('climaticos_*')):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f" - csv/{f.name} ({size_mb:.2f} MB)")
else:
    print("El directorio de CSVs no existe.")

print("\nArchivos Excel creados:")
if OUTPUT_XLSX_DIR.exists():
    for f in sorted(OUTPUT_XLSX_DIR.glob('climaticos_*')):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f" - xlsx/{f.name} ({size_mb:.2f} MB)")
else:
    print("El directorio de Excels no existe.")

## ℹ️ Nota sobre la Variabilidad en el Número de Estaciones (Idemas) por Lote

Es común observar que, aunque el inventario maestro de AEMET reporta aproximadamente **920 estaciones activas**, el número de registros diarios reales por lote no equivale exactamente a `920 estaciones × N días`.

Por ejemplo, en un lote de 15 días (como del 16 al 30 de mayo), se obtienen unos **12,176 registros**, lo que equivale a un promedio de **~812 estaciones diarias** con reporte de datos, en lugar de las 920 teóricas.

### ¿A qué se debe esta diferencia?
1. **Estaciones inactivas o en mantenimiento:** Algunas estaciones pueden sufrir fallos de comunicación, averías en sensores o estar en periodos de mantenimiento programado.
2. **Validación y desfase temporal de datos:** AEMET somete los datos climatológicos a controles de calidad antes de publicarlos en la API OpenData. Las estaciones de publicación reciente o secundarias pueden tardar más en volcar sus datos.
3. **Altas y bajas en la red:** El inventario incluye estaciones históricas o de alta reciente que no necesariamente reportan en la horquilla temporal consultada.
4. **Filtros de datos nulos:** Durante el proceso de ingesta, aquellos registros que no contienen lecturas válidas o están totalmente vacíos para los días del lote son descartados o no transmitidos por la propia API.